# 06 - Foundation model: Chronos-2 zero-shot

Chronos-2 (`amazon/chronos-2`) is a pre-trained time-series foundation model. It is used
here **zero-shot**: no parameter of the network is updated using the appliance data. The
model is given the observed history as context and asked for a predictive distribution over
the next 24 hours.

## What "zero-shot" means here, precisely

It is worth being exact, because the term is often used loosely.

- The model has **never seen this dataset** during our procedure. We do no fitting,
  fine-tuning or adaptation.
- It **has** seen the history of this series at inference time, as context.
- It was pre-trained by Amazon on a large corpus of time series, some real and some
  synthetic. We cannot verify that the Appliances Energy Prediction dataset, which is a
  well-known public UCI dataset, was excluded from that corpus. If it were included, the
  "zero-shot" claim would be weaker than it appears.

That last point is a genuine limitation and is stated as such in the report rather than
glossed over. It applies to any evaluation of a foundation model on a public benchmark
dataset.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, pipeline

from appliance_energy.models import foundation

hourly = data.load_hourly()
y = hourly[config.TARGET]
train, test = data.train_test_split(y)

## Requirements

Chronos needs `torch` and `chronos-forecasting`, and downloads model weights from the
Hugging Face hub the first time it runs:

```
pip install torch chronos-forecasting
```

If either is unavailable the pipeline falls back to a documented statistical model and
records `used_fallback: true` in `outputs/metrics/foundation_model_backend.json`. The
results below were produced with the real model; the JSON file confirms it.

## The context format

Chronos-2 takes a long DataFrame with an item id, a timestamp and a target column.

In [ ]:
context = foundation.make_context_frame(train)

print(context.shape)
context.head()

## A single forecast

Load the pipeline once and forecast the first 24-hour block.

In [ ]:
pipeline_obj = foundation.load_chronos2_pipeline()

median, lower, upper = foundation.chronos2_forecast(
    pipeline_obj,
    history=train,
    horizon=config.HORIZON,
    index=test.index[:config.HORIZON],
)

pd.DataFrame({
    "actual": test.iloc[:config.HORIZON],
    "chronos_10%": lower,
    "chronos_median": median,
    "chronos_90%": upper,
}).round(1)

## Rolling-origin backtest

The same procedure over all fourteen origins. At each origin the model receives every
observation up to that point as context; the weights never change, so it remains zero-shot
throughout.

In [ ]:
point, lower, upper, metadata = foundation.rolling_origin_foundation(
    y=y,
    n_origins=config.N_ORIGINS,
    horizon=config.HORIZON,
    backend="chronos2",
)

print("Backend used:", metadata.get("backend"))
print("Fell back:", metadata.get("used_fallback"))

evaluation.evaluate_forecast("chronos-2", test, point.reindex(test.index), train)

MASE 0.614, the best of any model in the study that is a valid 24-hour-ahead forecast.
Against the strongest benchmark at 0.712, that is a 14% improvement, achieved with no
fitting, no feature engineering and no tuning.

## But look at RMSE

In [ ]:
saved = pd.read_csv(config.METRICS_DIR / "model_comparison.csv")

saved.loc[saved["model"].isin([
    "foundation_model", "seasonal_mean_profile", "feature_model",
    "sarimax_target_only", "sarimax",
])].round(3)

Chronos has the **best MAE and the worst RMSE** of the serious models. Its bias is also
large and negative, at about -19 Wh, roughly three times that of any other model.

These three facts describe one behaviour: Chronos forecasts the typical hour very
accurately and systematically undershoots the peaks. MAE rewards that, because most hours
are typical. RMSE punishes it, because squared error is dominated by the few hours where
consumption spikes to 400 Wh and the forecast says 150.

Which metric matters depends on the application, and the report returns to this. For
choosing when to run a dishwasher, MAE is the right target. For sizing a battery or
avoiding a peak-demand charge, the errors that matter are exactly the ones RMSE weights.

## Prediction intervals

In [ ]:
interval_scores = pd.DataFrame([{
    "model": "chronos-2",
    "nominal": 0.80,
    "coverage": evaluation.coverage(test, lower.reindex(test.index), upper.reindex(test.index)),
    "average_width": evaluation.interval_width(lower.reindex(test.index), upper.reindex(test.index)),
}])

interval_scores.round(3)

This is where Chronos is most clearly ahead. Its 80% intervals achieve 77.7% empirical
coverage at an average width of 92 Wh. The SARIMAX intervals achieve 91% coverage at 181 Wh.

Chronos is therefore very slightly under-covering while SARIMAX substantially over-covers,
and Chronos does it with intervals **half as wide**. For a probabilistic forecast that is a
decisive difference: a sharp, well-calibrated interval supports decisions that a wide,
conservative one cannot.

The reason is structural. Chronos produces quantiles directly and is not constrained to a
symmetric Gaussian error distribution, so it can represent the asymmetry of a series that
has a hard floor near 50 Wh and a long right tail.

In [ ]:
fig = plotting.plot_intervals(test, point, lower, upper, label="Chronos-2", days=7)
fig

## Where it goes wrong

In [ ]:
error = point.reindex(test.index) - test

by_hour = pd.DataFrame({
    "MAE": error.abs().groupby(test.index.hour).mean(),
    "bias": error.groupby(test.index.hour).mean(),
}).round(1)

by_hour

Overnight the model is nearly exact: MAE around 3 Wh between 01:00 and 05:00. From 09:00
to 18:00 it rises to between 50 and 90 Wh, and the bias is strongly negative through the
peak hours.

The failure is entirely concentrated in the hours when the household is active and
consumption is driven by discretionary human decisions. No amount of pattern-matching on
the past recovers whether someone chose to cook at 18:00 on a particular Thursday.

## Discussion questions from the tutorial, answered for this dataset

**Is it genuinely zero-shot?** In our procedure, yes: no fitting occurred. With the caveat
above about the pre-training corpus, which we cannot inspect.

**Does the 80% interval achieve 80% coverage?** Close: 77.7%. Marginally narrow but far
better calibrated than the SARIMAX alternative.

**Where does it perform worst?** Daytime and evening peaks, with a systematic tendency to
undershoot. Overnight performance is close to perfect.

**How would known future covariates change things?** Chronos-2 does support covariates. We
did not use them, partly for a clean comparison against the target-only benchmarks and
partly because notebooks 04 and 05 both found covariates unhelpful for this series.